# 00 — Dataset Audit

**Mục tiêu:** kiểm tra tính hợp lệ của nguồn, schema, chất lượng và khả năng dùng cho phân loại bệnh.
**Câu hỏi:** dữ liệu là gì, có bao nhiêu lớp, bao nhiêu mẫu độc lập, có mẫu trùng/xung đột hoặc nguy cơ leakage không?

Chỉ Phase 1; không tiền xử lý để huấn luyện, không fit mô hình. Toàn bộ kết quả bên dưới tính từ CSV cấu hình.
Nếu dữ liệu thiếu/sai schema, chương trình dừng kèm chỉ dẫn thay vì tạo dữ liệu giả.

Runtime chung Colab/Drive: [hướng dẫn duy nhất](../docs/phase3_colab.md). Cấu hình ở `configs/runtime.json`; source từ Drive, thực thi trong `/content`. Outputs đã lưu là lịch sử, không phải bằng chứng đã chạy trên Colab.

In [2]:
import runpy, platform
from pathlib import Path
if platform.system() != 'Linux' or not Path('/content').is_dir():
    raise RuntimeError('Chọn Google Colab kernel; /content phải là ổ tạm của runtime.')
DRIVE_ROOT = Path('/content/drive/MyDrive/ML_Projects/disease-prediction-ml')  # Locator sinh từ configs/runtime.json
entry = DRIVE_ROOT / 'src/data/colab_runtime.py'
if not entry.is_file():
    raise FileNotFoundError(f'Không thấy {entry}. Mount Drive và Publish source bằng prepare_colab_source.ps1; không upload script riêng.')
EXPECTED_RUNTIME_PAYLOAD = globals().get('EXPECTED_RUNTIME_PAYLOAD') or input('Paste RUNTIME_PAYLOAD_SHA256 from local Publish: ').strip()
RUNTIME = runpy.run_path(str(entry))['bootstrap_runtime'](DRIVE_ROOT, phase=1, expected_payload_sha256=EXPECTED_RUNTIME_PAYLOAD)
ROOT, runtime_paths = RUNTIME.paths.project_root, RUNTIME.paths

Stage A PASS | Colab Python 3.13.16 | Drive=/content/drive/MyDrive/ML_Projects/disease-prediction-ml | runtime/Drive read-write OK
Stage B PASS | imports=/content/disease-prediction-ml-runtime | source fingerprint=95b01d6ea67d68f9bbc48b647ca15b3c38219baac15639ae78637f184806bd61


In [3]:
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from src.data.io import read_config, save_json
from src.data.loader import dataset_path, load_dataset, prepare_dataset, sha256_file
from src.data.audit import audit_dataset
config = read_config(ROOT)
np.random.seed(config["seed"])
source_path = dataset_path(runtime_paths, config)
path = prepare_dataset(runtime_paths, source_path)
df = load_dataset(path, config["target"])
OUTPUT = runtime_paths.new_run()
save_json(OUTPUT.metadata_dir / "runtime.json", {
    **runtime_paths.as_dict(), "source_csv": str(source_path), "read_csv": str(path),
    "source_sha256": sha256_file(source_path), "run_id": OUTPUT.run_id,
})
print(f"Runtime: {runtime_paths.runtime}; CSV nguồn: {source_path}")
print(f"Run ID: {OUTPUT.run_id}")
print(f"Báo cáo mới: {OUTPUT.reports_dir}; metadata: {OUTPUT.metadata_dir}")
print(f"Đọc toàn bộ {len(df):,} dòng, {df.shape[1]} cột từ {path.name}. Raw chỉ được đọc.")

Runtime: colab; CSV nguồn: /content/drive/MyDrive/ML_Projects/disease-prediction-ml/data/raw/Disease and symptoms dataset.csv
Run ID: phase1-20261027T135449072864Z-41b427b5
Báo cáo mới: /content/drive/MyDrive/ML_Projects/disease-prediction-ml/reports/runs/phase1-20261027T135449072864Z-41b427b5; metadata: /content/drive/MyDrive/ML_Projects/disease-prediction-ml/artifacts/metadata/runs/phase1-20261027T135449072864Z-41b427b5
Đọc toàn bộ 246,945 dòng, 378 cột từ Disease and symptoms dataset.csv. Raw chỉ được đọc.


## 1. Nguồn, cấu trúc và phạm vi xác minh
Nguồn ứng viên Mendeley và giới hạn xác minh được lưu trong `configs/dataset_source.json`.
Tách nhãn khỏi đặc trưng; kiểm tra giá trị trước khi tối ưu dtype trong RAM. Không thay đổi tệp raw.

In [4]:
import json
source = json.loads((ROOT / config["source_metadata"]).read_text(encoding="utf-8"))
display(Markdown(f"Nguồn ứng viên: [{source['title']}]({source['url']}), DOI `{source['doi']}`; version {source['version']}."))
display(pd.DataFrame({"thuộc tính": list(source), "giá trị": [str(v) for v in source.values()]}))
display(df.head(5).iloc[:, :10])
audit = audit_dataset(df, config)
s = audit["summary"]
display(pd.DataFrame([{"chỉ số": k, "giá trị": v} for k, v in s.items()]))

Nguồn ứng viên: [Disease and symptoms dataset 2023](https://data.mendeley.com/datasets/2cxccsxydc/1), DOI `10.17632/2cxccsxydc.1`; version 1.

,thuộc tính,giá trị
0,title,Disease and symptoms dataset 2023
1,url,https://data.mendeley.com/datasets/2cxccsxydc/1
2,doi,10.17632/2cxccsxydc.1
3,contributor,Bran Stark
4,version,1
5,published,2025-03-03
6,accessed,2026-10-21
7,license,CC BY 4.0
8,reported_classes,773
9,reported_features,377


,diseases,anxiety and nervousness,depression,shortness of breath,depressive or psychotic symptoms,sharp chest pain,dizziness,insomnia,abnormal involuntary movements,chest tightness
0,panic disorder,1,0,1,1,0,0,0,0,1
1,panic disorder,0,0,1,1,0,1,1,0,0
2,panic disorder,1,1,1,1,0,1,1,0,0
3,panic disorder,1,0,0,1,0,1,1,1,0
4,panic disorder,1,1,0,0,0,0,1,1,1


,chỉ số,giá trị
0,rows,246945
1,columns,378
2,features,377
3,target,diseases
4,classes,773
5,missing_cells,0
6,missing_target_rows,0
7,blank_target_rows,0
8,invalid_binary_cells,0
9,binary_complete,True


## 2. Kiểu dữ liệu, giá trị thiếu, hằng và miền hợp lệ
Ngưỡng gần hằng xét tỷ lệ giá trị phổ biến ≥99%; cột hằng là tập con.
Phân biệt dấu hiệu nhị phân quan sát được với cách mã hóa đã được xác minh về lâm sàng.

In [5]:
display(audit["missing"].sort_values("missing_count", ascending=False).head(15))
display(audit["features"].head(15))
display(audit["features"].query("constant or invalid_binary_count > 0"))
display(audit["alias_candidates"])
display(audit["duplicate_columns"])
display(audit["label_collisions"])
display(Markdown(f"Có **{s['missing_cells']:,}** ô thiếu, **{s['invalid_binary_cells']:,}** ô ngoài miền 0/1 và **{s['constant_features']}** cột hằng. Cờ này chưa quyết định loại cột; danh sách đầy đủ sẽ được lưu."))

,column,missing_count,missing_pct,blank_string_count,invalid_or_blank_count
377,neck weakness,0,0.0,0,0
0,diseases,0,0.0,0,0
1,anxiety and nervousness,0,0.0,0,0
2,depression,0,0.0,0,0
3,shortness of breath,0,0.0,0,0
4,depressive or psychotic symptoms,0,0.0,0,0
5,sharp chest pain,0,0.0,0,0
6,dizziness,0,0.0,0,0
7,insomnia,0,0.0,0,0
8,abnormal involuntary movements,0,0.0,0,0


,feature,input_dtype,memory_dtype,meaning,n_unique_nonmissing,missing_count,missing_pct,is_binary_observed,invalid_binary_count,invalid_examples,min,max,dominant_fraction,constant,near_constant,present_count,present_fraction
0,anxiety and nervousness,int64,uint8,"Symptom indicator; English name, no verified c...",2,0,0.0,True,0,,0.0,1.0,0.960765,False,False,9689,0.039235
1,depression,int64,uint8,"Symptom indicator; English name, no verified c...",2,0,0.0,True,0,,0.0,1.0,0.957254,False,False,10556,0.042746
2,shortness of breath,int64,uint8,"Symptom indicator; English name, no verified c...",2,0,0.0,True,0,,0.0,1.0,0.913560,False,False,21346,0.086440
3,depressive or psychotic symptoms,int64,uint8,"Symptom indicator; English name, no verified c...",2,0,0.0,True,0,,0.0,1.0,0.938999,False,False,15064,0.061001
4,sharp chest pain,int64,uint8,"Symptom indicator; English name, no verified c...",2,0,0.0,True,0,,0.0,1.0,0.902748,False,False,24016,0.097252
5,dizziness,int64,uint8,"Symptom indicator; English name, no verified c...",2,0,0.0,True,0,,0.0,1.0,0.930057,False,False,17272,0.069943
6,insomnia,int64,uint8,"Symptom indicator; English name, no verified c...",2,0,0.0,True,0,,0.0,1.0,0.960590,False,False,9732,0.039410
7,abnormal involuntary movements,int64,uint8,"Symptom indicator; English name, no verified c...",2,0,0.0,True,0,,0.0,1.0,0.959428,False,False,10019,0.040572
8,chest tightness,int64,uint8,"Symptom indicator; English name, no verified c...",2,0,0.0,True,0,,0.0,1.0,0.962129,False,False,9352,0.037871
9,palpitations,int64,uint8,"Symptom indicator; English name, no verified c...",2,0,0.0,True,0,,0.0,1.0,0.975124,False,False,6143,0.024876


,feature,input_dtype,memory_dtype,meaning,n_unique_nonmissing,missing_count,missing_pct,is_binary_observed,invalid_binary_count,invalid_examples,min,max,dominant_fraction,constant,near_constant,present_count,present_fraction
33,pus in sputum,int64,uint8,"Symptom indicator; English name, no verified c...",1,0,0.0,True,0,,0.0,0.0,1.0,True,True,0,0.0
134,underweight,int64,uint8,"Symptom indicator; English name, no verified c...",1,0,0.0,True,0,,0.0,0.0,1.0,True,True,0,0.0
172,arm cramps or spasms,int64,uint8,"Symptom indicator; English name, no verified c...",1,0,0.0,True,0,,0.0,0.0,1.0,True,True,0,0.0
176,abnormal appearing tongue,int64,uint8,"Symptom indicator; English name, no verified c...",1,0,0.0,True,0,,0.0,0.0,1.0,True,True,0,0.0
194,pallor,int64,uint8,"Symptom indicator; English name, no verified c...",1,0,0.0,True,0,,0.0,0.0,1.0,True,True,0,0.0
207,shoulder cramps or spasms,int64,uint8,"Symptom indicator; English name, no verified c...",1,0,0.0,True,0,,0.0,0.0,1.0,True,True,0,0.0
208,joint stiffness or tightness,int64,uint8,"Symptom indicator; English name, no verified c...",1,0,0.0,True,0,,0.0,0.0,1.0,True,True,0,0.0
217,eye strain,int64,uint8,"Symptom indicator; English name, no verified c...",1,0,0.0,True,0,,0.0,0.0,1.0,True,True,0,0.0
224,pus in urine,int64,uint8,"Symptom indicator; English name, no verified c...",1,0,0.0,True,0,,0.0,0.0,1.0,True,True,0,0.0
225,abnormal size or shape of ear,int64,uint8,"Symptom indicator; English name, no verified c...",1,0,0.0,True,0,,0.0,0.0,1.0,True,True,0,0.0


,feature,base_name,equal_values,different_rows
0,regurgitation.1,regurgitation,False,1680


,group_id,n_features,features,constant
0,33,49,pus in sputum | underweight | arm cramps or sp...,True


,normalized,original_labels


Có **0** ô thiếu, **0** ô ngoài miền 0/1 và **49** cột hằng. Cờ này chưa quyết định loại cột; danh sách đầy đủ sẽ được lưu.

## 3. Mẫu trùng và nhãn xung đột
Vector nhị phân được pack lossless rồi so sánh byte bằng nhau; không dùng hash xác suất để kết luận trùng.
“Dòng dư” bỏ bản đầu; “dòng thuộc nhóm lặp” tính cả bản đầu. Nhóm đặc trưng bỏ qua nhãn để phát hiện xung đột.

In [6]:
groups = audit["pattern_groups"]
display(groups.sort_values(["distinct_labels", "rows"], ascending=False).head(15))
conflict_ids = groups.loc[groups["distinct_labels"] > 1, "pattern_id"].head(5)
display(audit["pattern_labels"].query("pattern_id in @conflict_ids"))
display(audit["class_patterns"].sort_values(["unique_patterns", "total_rows"]).head(20))
display(Markdown(f"Có **{s['unique_feature_vectors']:,}** vector khác nhau từ **{s['rows']:,}** dòng; **{s['exact_duplicate_excess_rows']:,}** dòng dư nếu xét cả nhãn; **{s['conflicting_pattern_groups']:,}** nhóm có nhiều nhãn. Chia theo dòng có thể đưa cùng triệu chứng sang cả train và test."))

,pattern_id,rows,distinct_labels,majority_label_rows,missing_label_rows
4260,4260,27,26,2,0
14853,14853,30,24,3,0
4087,4087,30,22,6,0
27759,27759,21,21,1,0
9789,9789,22,20,2,0
331,331,19,19,1,0
10449,10449,18,17,2,0
1540,1540,18,15,4,0
6606,6606,16,15,2,0
12710,12710,22,14,6,0


,pattern_id,disease,rows,missing_label_rows
6,6,panic disorder,1,0
43,43,panic disorder,1,0
57,57,panic disorder,1,0
71,71,panic disorder,1,0
76,76,panic disorder,1,0
15084,6,hypertrophic obstructive cardiomyopathy (hocm),1,0
21576,6,marijuana abuse,1,0
33317,6,primary insomnia,1,0
58300,57,alcohol intoxication,1,0
59341,43,anxiety,1,0


,disease,unique_patterns,total_rows,rows_in_conflicting_patterns
131,chronic ulcer,1,1,0
178,diabetes,1,1,0
256,foreign body in the nose,1,1,0
282,gas gangrene,1,1,0
310,heat stroke,1,1,0
325,high blood pressure,1,1,1
332,huntington disease,1,1,0
340,hypergammaglobulinemia,1,1,1
406,kaposi sarcoma,1,1,1
465,myocarditis,1,1,1


Có **169,888** vector khác nhau từ **246,945** dòng; **57,298** dòng dư nếu xét cả nhãn; **12,634** nhóm có nhiều nhãn. Chia theo dòng có thể đưa cùng triệu chứng sang cả train và test.

## 4. Rò rỉ nhãn và tính phù hợp ban đầu
Kiểm tra tên cột chỉ là sàng lọc. Feature đặc hiệu không tự chứng minh leakage hay tính đúng lâm sàng.
Cơ chế sinh dữ liệu, các cột đồng nghĩa, mã bệnh lẫn trong đặc trưng và trùng bệnh nhân cần tài liệu nguồn.
Nhãn có đủ dòng chưa chắc có đủ vector độc lập để đánh giá.

In [7]:
display(audit["class_distribution"].head(15))
display(Markdown(f"**Đánh giá ban đầu: {s['recommendation']}**. Có {s['classes_with_lt5_rows']} lớp dưới 5 dòng và {s['classes_with_lt5_patterns']} lớp dưới 5 vector. EDA tiếp theo sẽ xét phân bố, quan hệ triệu chứng và điều kiện chia group."))
print("Ứng viên leakage theo tên:", s["name_leakage_candidates"])

,disease,samples,fraction_all_rows
0,cystitis,1219,0.004936
1,nose disorder,1218,0.004932
2,vulvodynia,1218,0.004932
3,complex regional pain syndrome,1217,0.004928
4,spondylosis,1216,0.004924
5,vaginal cyst,1215,0.004920
6,esophagitis,1215,0.004920
7,peripheral nerve disorder,1215,0.004920
8,hypoglycemia,1215,0.004920
9,conjunctivitis due to allergy,1215,0.004920


**Đánh giá ban đầu: CONDITIONAL GO**. Có 52 lớp dưới 5 dòng và 115 lớp dưới 5 vector. EDA tiếp theo sẽ xét phân bố, quan hệ triệu chứng và điều kiện chia group.

Ứng viên leakage theo tên: []


## 5. Xuất báo cáo và fingerprint
Báo cáo ghi nguồn ứng viên, giới hạn xác minh, từng định nghĩa mẫu trùng và khuyến nghị ban đầu.
CSV đầu vào được băm trước/sau; bảng pattern_id phục vụ thiết kế chia nhóm ở giai đoạn sau, chưa chia dữ liệu.

In [8]:
from src.data.reporting import export_audit
from src.data.io import sha256_file
manifest = export_audit(ROOT, path, config, df, audit, output=OUTPUT)
assert sha256_file(path) == manifest["sha256"], "Raw thay đổi trong lúc audit!"
display(Markdown((OUTPUT.reports_dir / "dataset_audit.md").read_text(encoding="utf-8")))
print(f"Báo cáo: {OUTPUT.reports_dir}; fingerprint: {OUTPUT.metadata_dir}")

# Báo cáo Dataset Audit

Kết quả tính trên toàn bộ `Disease and symptoms dataset.csv`; không lấy mẫu. SHA-256: `8de90603ebada467fc2703db2ee4292856e52d64e94982ef926df22fa0c21320`.
Số dòng dữ liệu trong bảng ánh xạ là 1-based, không tính header CSV. `pattern_id` chỉ có ý nghĩa kèm fingerprint và thứ tự cột của tệp này.

## Nguồn và phạm vi xác minh

Nguồn ứng viên: [Disease and symptoms dataset 2023](https://data.mendeley.com/datasets/2cxccsxydc/1), DOI `10.17632/2cxccsxydc.1`, contributor Bran Stark, version 1, công bố 2025-03-03, giấy phép CC BY 4.0; tra cứu ngày 2026-10-21.
Tên tệp, schema và số lớp/đặc trưng khớp mô tả. Chưa có checksum nhà xuất bản hoặc lịch sử tải xuống để xác nhận đây là bản gốc không chỉnh sửa.
Nhà xuất bản mô tả dữ liệu sinh nhân tạo. Không có căn cứ xem mỗi dòng là một bệnh nhân thật; tuyên bố về tỷ lệ mắc bệnh và độ nặng chưa được kiểm chứng độc lập.
Trang nguồn không cung cấp quy trình kiểm định lâm sàng hoặc mã sinh dữ liệu đủ để tái tạo trong nội dung đã xem.

## Kết quả cấu trúc và chất lượng

| Chỉ số | Giá trị |
| --- | --- |
| rows | 246945 |
| columns | 378 |
| features | 377 |
| target | diseases |
| classes | 773 |
| missing_cells | 0 |
| missing_target_rows | 0 |
| blank_target_rows | 0 |
| invalid_binary_cells | 0 |
| binary_complete | True |
| constant_features | 49 |
| near_constant_features_including_constant | 227 |
| exact_duplicate_excess_rows | 57298 |
| rows_in_exact_duplicate_groups | 94067 |
| unique_feature_vectors | 169888 |
| repeated_pattern_groups | 44287 |
| rows_in_repeated_patterns | 121344 |
| repeated_pattern_excess_rows | 77057 |
| conflicting_pattern_groups | 12634 |
| rows_in_conflicting_patterns | 39728 |
| min_class_samples | 1 |
| max_class_samples | 1219 |
| median_class_samples | 168.0 |
| classes_with_lt5_rows | 52 |
| classes_with_lt5_patterns | 115 |
| classes_with_one_pattern | 46 |
| name_leakage_candidates | [] |
| normalized_label_collisions | 0 |
| empirical_pattern_majority_ceiling | 0.9162080625240439 |
| recommendation | CONDITIONAL GO |

`exact_duplicate_excess_rows` = số dòng dư sau khi giữ một dòng cho mỗi cặp (vector, nhãn).
`repeated_pattern_excess_rows` = số dòng dư sau khi giữ một dòng cho mỗi vector, bỏ qua nhãn.
`rows_in_repeated_patterns` tính tất cả dòng thuộc nhóm vector có kích thước >1, gồm cả bản đầu tiên.
`near_constant_features_including_constant` dùng tần suất giá trị chiếm ưu thế ≥99%; đây chỉ là cờ mô tả, chưa loại cột.

## Nhãn, ý nghĩa đặc trưng và leakage

Nhãn `diseases` được tách khỏi X khi phân tích vector triệu chứng. Bảng `audit_features.csv` chứa tên, dtype gốc, miền giá trị, tỷ lệ thiếu, tần suất, cờ hằng và gần hằng của từng cột.
Từ điển chuyên môn chưa được xác minh: tên cột tiếng Anh gợi ý triệu chứng, không chứng minh độ tin cậy y khoa.
Các cột nghi vấn do tên: []. Không có cờ theo tên không đồng nghĩa không có leakage.
`audit_alias_candidates.csv` kiểm tra hậu tố như `.1`; không tự gộp các tên giống nhau. `audit_duplicate_columns.csv` chỉ báo các cột có vector giá trị giống hệt, gồm nhóm cột hằng.
Không có patient ID, thời gian hay cơ sở khám để kiểm tra trùng bệnh nhân hoặc chia theo bệnh viện/thời gian. Quan hệ đặc trưng–nhãn quá đặc hiệu cần xét cùng cơ chế sinh nhân tạo, chưa thể kết luận leakage nhân quả.

## Đánh giá ban đầu: CONDITIONAL GO

Có cấu trúc phân loại có giám sát với 773 lớp. Tính đại diện phải đánh giá bằng **số vector độc lập theo lớp**, không chỉ số dòng.
Có 12,634 vector mang nhiều nhãn và 115 lớp có dưới 5 vector.
Mức majority theo vector trên toàn bộ tệp = 0.9162080625240439; đây là giới hạn ghi nhớ thực nghiệm của bộ phân loại tất định trên chính các dòng đã quan sát, không phải độ chính xác kiểm định hoặc giới hạn quần thể.
Kết luận ban đầu dành cho nghiên cứu học phần; các điều kiện cụ thể và phân tích phân tách sẽ được cập nhật trong EDA. Không suy rộng sang chẩn đoán lâm sàng.

## Ngăn leakage và hướng xử lý tiếp theo

Không chia ngẫu nhiên theo dòng khi vector lặp. Giữ mọi dòng có cùng vector trong cùng một group cho mọi phép chia và CV, kể cả khi khác nhãn.
Kiểm tra số group của từng lớp trước khi đề xuất số fold. Lớp có một group không thể đồng thời có train và test độc lập; không tự nhân bản, bỏ lớp hoặc hợp nhất bệnh.
Chưa chọn quy tắc cho xung đột nhãn; cần quyết định trên tập phát triển và tài liệu nguồn, báo cáo ảnh hưởng nếu loại/gộp.
Imputation, scaling, feature selection và PCA sau này chỉ được fit trên train/CV training fold. Test cuối khóa riêng, không dùng chọn siêu tham số.

## Giao diện tiếng Việt

Chưa có ánh xạ Việt–Anh, kiểm định từ đồng nghĩa/phủ định hay dữ liệu hội thoại. Giá trị 0 không chứng minh triệu chứng đã được hỏi và phủ nhận; không tự mã hóa mọi thông tin chưa nhắc đến thành âm tính.
Các cặp gần nghĩa và hậu tố `.1` cần được rà soát trước khi xây từ điển. ML sẽ dự đoán nhãn; LLM chỉ hỗ trợ hiểu tiếng Việt, hỏi rõ và giải thích. Phase 1 chưa triển khai giao diện hoặc chatbot.


Báo cáo: /content/drive/MyDrive/ML_Projects/disease-prediction-ml/reports/runs/phase1-20261027T135449072864Z-41b427b5; fingerprint: /content/drive/MyDrive/ML_Projects/disease-prediction-ml/artifacts/metadata/runs/phase1-20261027T135449072864Z-41b427b5
